In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [ ]:
query = """

WITH legs AS (
    -- entity is buyer = receiver of currency1 (ESMA REFIT GL §435: leg-1 TAKE), so long currency1
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'long' ELSE 'short' END AS direction
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND buyer_nfc = 1
    AND buyer_id NOT IN ('7LTWFZYICNSX8D621K86', 'R0MUWSFPU8MPRO8K5P83', 'K8MS7FD7N5Z2WQ51AZ71', '5493006QMFDDMYWIAM13', '529900ODI3047E2LIV03')
    AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
    UNION ALL
    -- entity is seller = payer of currency1 (leg-1 MAKE), so short currency1
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'short' ELSE 'long' END AS direction
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND seller_nfc = 1
    AND notional_eur <= 1e11
    AND seller_id NOT IN ('7LTWFZYICNSX8D621K86', 'R0MUWSFPU8MPRO8K5P83', 'K8MS7FD7N5Z2WQ51AZ71', '5493006QMFDDMYWIAM13', '529900ODI3047E2LIV03')
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
)
SELECT
    reference_period,
    SUM(CASE WHEN direction = 'long'  THEN notional_eur ELSE 0 END) AS long_eur,
    SUM(CASE WHEN direction = 'short' THEN notional_eur ELSE 0 END) AS short_eur
FROM legs
GROUP BY reference_period
ORDER BY reference_period

"""

ts = pd.read_sql_query(query, cnxn)

In [ ]:
ts['reference_period'] = pd.to_datetime(ts['reference_period'])
ts = ts.set_index('reference_period').sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(ts.index, ts['long_eur'] / 1e9, label='Long Foreign Currency')
ax.plot(ts.index, ts['short_eur'] / 1e9, label='Short Foreign Currency')
ax.set_xlabel('Reference period')
ax.set_ylabel('Outstanding notional (EUR bn)')
ax.set_title('Outstanding forward positions')
ax.legend()
plt.show()

In [ ]:
banks = "'7LTWFZYICNSX8D621K86', 'R0MUWSFPU8MPRO8K5P83', 'K8MS7FD7N5Z2WQ51AZ71', '5493006QMFDDMYWIAM13', '529900ODI3047E2LIV03'"
asian = "'JPY', 'CNY', 'CNH', 'HKD', 'SGD', 'KRW', 'TWD', 'INR', 'THB', 'IDR', 'MYR', 'PHP', 'PKR', 'VND', 'BDT', 'LKR', 'KZT', 'UZS'"
north_american = "'USD', 'CAD', 'MXN'"
european = "'GBP', 'CHF', 'SEK', 'NOK', 'DKK', 'PLN', 'CZK', 'HUF', 'RON', 'BGN', 'HRK', 'RSD', 'ISK', 'UAH'"

query = f"""

WITH legs AS (
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'long' ELSE 'short' END AS direction,
           CASE WHEN currency1 = 'EUR' THEN currency2 ELSE currency1 END AS foreign_ccy
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND buyer_nfc = 1
      AND buyer_id NOT IN ({banks})
      AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
    UNION ALL
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'short' ELSE 'long' END AS direction,
           CASE WHEN currency1 = 'EUR' THEN currency2 ELSE currency1 END AS foreign_ccy
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND seller_nfc = 1
      AND seller_id NOT IN ({banks})
      AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
),
classified AS (
    SELECT reference_period, notional_eur, direction,
           CASE WHEN foreign_ccy IN ({asian})          THEN 'Asia'
                WHEN foreign_ccy IN ({north_american})  THEN 'North America'
                WHEN foreign_ccy IN ({european})        THEN 'Europe'
                ELSE 'Other' END AS region
    FROM legs
),
n AS (SELECT COUNT(DISTINCT reference_period) AS n_periods FROM classified)
-- average outstanding notional per reporting date, by region and direction
SELECT region, direction, SUM(notional_eur) / MAX(n_periods) AS avg_eur
FROM classified CROSS JOIN n
GROUP BY region, direction
ORDER BY region, direction

"""

reg = pd.read_sql_query(query, cnxn)

In [ ]:
regions = ['Asia', 'North America', 'Europe', 'Other']
piv = (reg.pivot(index='region', columns='direction', values='avg_eur')
          .reindex(index=regions, columns=['long', 'short'], fill_value=0))

x = np.arange(len(regions))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - w / 2, piv['long'] / 1e9, w, label='Long Foreign Currency')
ax.bar(x + w / 2, piv['short'] / 1e9, w, label='Short Foreign Currency')
ax.set_xticks(x)
ax.set_xticklabels(regions)
ax.set_ylabel('Avg. outstanding notional (EUR bn)')
ax.set_title('Forward positions by currency region')
ax.legend()
plt.show()

In [ ]:
start = '2024-04-26'  # REFIT period only (exclude the pre-2019 EMIR snapshots)

query = f"""

WITH legs AS (
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'long' ELSE 'short' END AS direction,
           CASE WHEN currency1 = 'EUR' THEN currency2 ELSE currency1 END AS foreign_ccy
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND buyer_nfc = 1
      AND buyer_id NOT IN ({banks})
      AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
      AND reference_period >= '{start}'
      AND reference_period NOT IN ('2025-04-18', '2025-04-21')
    UNION ALL
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'short' ELSE 'long' END AS direction,
           CASE WHEN currency1 = 'EUR' THEN currency2 ELSE currency1 END AS foreign_ccy
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND seller_nfc = 1
      AND seller_id NOT IN ({banks})
      AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
      AND reference_period >= '{start}'
      AND reference_period NOT IN ('2025-04-18', '2025-04-21')
),
classified AS (
    SELECT reference_period, notional_eur, direction,
           CASE WHEN foreign_ccy = 'USD'        THEN 'USD'
                WHEN foreign_ccy IN ({asian})    THEN 'Asia'
                WHEN foreign_ccy IN ({european}) THEN 'Europe' END AS grp
    FROM legs
)
-- net (long - short) outstanding notional per day and group
SELECT reference_period, grp,
       SUM(CASE WHEN direction = 'long' THEN notional_eur ELSE -notional_eur END) AS net_eur
FROM classified
WHERE grp IS NOT NULL
GROUP BY reference_period, grp
ORDER BY reference_period, grp

"""

es = pd.read_sql_query(query, cnxn)

In [ ]:
event = pd.Timestamp('2025-04-02')  # Liberation Day tariff announcement
base_window = 30                    # days before the event used as the baseline
plot_window = 30                    # days shown on each side of the event

net = es.pivot(index='reference_period', columns='grp', values='net_eur')
net.index = pd.to_datetime(net.index)
net = net.sort_index()

# standardise each series against its pre-event window (mean 0, sd 1) so the
# differently-sized series are comparable and all sit at the same level pre-event
pre = net[(net.index >= event - pd.Timedelta(days=base_window)) & (net.index < event)]
z = (net - pre.mean()) / pre.std()
z = z[(z.index >= event - pd.Timedelta(days=plot_window)) & (z.index <= event + pd.Timedelta(days=plot_window))]

fig, ax = plt.subplots(figsize=(12, 6))
for grp in ['USD', 'Asia', 'Europe']:
    ax.plot(z.index, z[grp], label=grp)
ax.axvline(event, color='black', linestyle='--', linewidth=1, label='Liberation Day')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_xlabel('Reference period')
ax.set_ylabel('Net position (std. dev. from pre-event mean)')
ax.set_title('Net forward positions around Liberation Day')
ax.set_ylim(-10, 10)
ax.legend()
plt.show()

In [ ]:
# Flow = newly initiated forwards, each contract counted once at its first
# appearance (no UTI, so the contract key uses economic attributes)

start = '2024-04-26'  # REFIT period only (exclude the pre-2019 EMIR snapshots)

query = f"""

WITH legs AS (
    SELECT 'B' AS side, buyer_id AS cp1, seller_id AS cp2,
           reference_period, effective_date, maturity_date, currency1, currency2,
           CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'long' ELSE 'short' END AS direction,
           CASE WHEN currency1 = 'EUR' THEN currency2 ELSE currency1 END AS foreign_ccy
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND buyer_nfc = 1
      AND buyer_id NOT IN ({banks})
      AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
      AND reference_period >= '{start}'
      AND reference_period NOT IN ('2025-04-18', '2025-04-21')
    UNION ALL
    SELECT 'S' AS side, buyer_id AS cp1, seller_id AS cp2,
           reference_period, effective_date, maturity_date, currency1, currency2,
           CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'short' ELSE 'long' END AS direction,
           CASE WHEN currency1 = 'EUR' THEN currency2 ELSE currency1 END AS foreign_ccy
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND seller_nfc = 1
      AND seller_id NOT IN ({banks})
      AND notional_eur <= 1e11
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
      AND reference_period >= '{start}'
      AND reference_period NOT IN ('2025-04-18', '2025-04-21')
),
tagged AS (
    SELECT effective_date, direction, notional_eur,
           CASE WHEN foreign_ccy = 'USD'        THEN 'USD'
                WHEN foreign_ccy IN ({asian})    THEN 'Asia'
                WHEN foreign_ccy IN ({european}) THEN 'Europe' END AS grp,
           ROW_NUMBER() OVER (PARTITION BY side, cp1, cp2, effective_date, maturity_date,
                                           currency1, currency2, notional_eur
                              ORDER BY reference_period) AS rn
    FROM legs
)
-- net (long - short) new forward notional per effective date and group
SELECT effective_date, grp,
       SUM(CASE WHEN direction = 'long' THEN notional_eur ELSE -notional_eur END) AS net_eur
FROM tagged
WHERE grp IS NOT NULL AND rn = 1
GROUP BY effective_date, grp
ORDER BY effective_date, grp

"""

flow = pd.read_sql_query(query, cnxn)

In [ ]:
event = pd.Timestamp('2025-04-02')  # Liberation Day tariff announcement
base_window = 60                    # days before the event used as the baseline
plot_window = 60                    # days shown on each side of the event

net = flow.pivot(index='effective_date', columns='grp', values='net_eur')
net.index = pd.to_datetime(net.index)
net = net.sort_index().fillna(0)  # a day with no new contracts is zero flow, not missing

# standardise each series against its pre-event window (mean 0, sd 1)
pre = net[(net.index >= event - pd.Timedelta(days=base_window)) & (net.index < event)]
z = (net - pre.mean()) / pre.std()
z = z[(z.index >= event - pd.Timedelta(days=plot_window)) & (z.index <= event + pd.Timedelta(days=plot_window))]

fig, ax = plt.subplots(figsize=(12, 6))
for grp in ['USD', 'Asia', 'Europe']:
    ax.plot(z.index, z[grp], label=grp)
ax.axvline(event, color='black', linestyle='--', linewidth=1, label='Liberation Day')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_xlabel('Effective date')
ax.set_ylabel('New net forward flow (std. dev. from pre-event mean)')
ax.set_title('New forward flow around Liberation Day')
ax.legend()
plt.show()

In [ ]:
# Cumulative net new flow, re-based to 0 just before the event. Unlike the stock,
# this sums inceptions only (ignores maturities), so it isolates new positioning.
cum = net[(net.index >= event - pd.Timedelta(days=plot_window)) &
          (net.index <= event + pd.Timedelta(days=plot_window))].cumsum()
cum = (cum - cum[cum.index < event].iloc[-1]) / 1e9

fig, ax = plt.subplots(figsize=(12, 6))
for grp in ['USD', 'Asia', 'Europe']:
    ax.plot(cum.index, cum[grp], label=grp)
ax.axvline(event, color='black', linestyle='--', linewidth=1, label='Liberation Day')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_xlabel('Effective date')
ax.set_ylabel('Cumulative net new forward flow since event (EUR bn)')
ax.set_title('Cumulative new forward flow around Liberation Day')
ax.legend()
plt.show()